# Chapter 8: PDEs and numerical methods

A reproducible companion laboratory. Run all cells in order. Figures are created by the accompanying `chapter08_figures.py`.

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.integrate import quad

## 1. Symbolic verification of the heat equation

In [ ]:
x, t, alpha = sp.symbols("x t alpha", positive=True)
u = sp.exp(-alpha*sp.pi**2*t)*sp.sin(sp.pi*x)
sp.simplify(sp.diff(u,t)-alpha*sp.diff(u,x,2))

## 2. Transport along characteristics

In [ ]:
xx=np.linspace(-3,5,500)
for time in [0,.4,.8,1.2]:
    plt.plot(xx,np.exp(-4*(xx-2*time)**2),label=f"t={time}")
plt.legend();plt.xlabel("x");plt.ylabel("u");plt.show()

## 3. Fourier sine coefficients

In [ ]:
def coefficient(k):
    return 2*quad(lambda z:z*(1-z)*np.sin(k*np.pi*z),0,1)[0]
[(k,coefficient(k),4*(1-(-1)**k)/(k**3*np.pi**3)) for k in range(1,7)]

## 4. Stable FTCS refinement

In [ ]:
def ftcs(n,alpha=.1,T=.2):
    xx=np.linspace(0,1,n);h=xx[1]-xx[0]
    steps=int(np.ceil(T*alpha/(.45*h*h)));dt=T/steps;r=alpha*dt/h**2
    y=np.sin(np.pi*xx)
    for _ in range(steps):
        z=y.copy();z[1:-1]=y[1:-1]+r*(y[:-2]-2*y[1:-1]+y[2:]);y=z
    return xx,y,r
results=[]
for n in [11,21,41,81,161]:
    xx,y,r=ftcs(n)
    err=np.max(np.abs(y-np.exp(-.1*np.pi**2*.2)*np.sin(np.pi*xx)))
    results.append((xx[1]-xx[0],err))
    print(n,"r=",round(r,3),"error=",err)
h,e=np.array(results).T
print("observed order",np.polyfit(np.log(h),np.log(e),1)[0])
plt.loglog(h,e,"o-");plt.xlabel("h");plt.ylabel("maximum error");plt.show()

## 5. Jacobi solution of Laplace's equation

In [ ]:
n=41;xx=np.linspace(0,1,n);v=np.zeros((n,n));v[-1,:]=np.sin(np.pi*xx)
for step in range(50000):
    w=v.copy()
    w[1:-1,1:-1]=.25*(v[2:,1:-1]+v[:-2,1:-1]+v[1:-1,2:]+v[1:-1,:-2])
    if np.max(np.abs(w-v))<1e-7:
        v=w;break
    v=w
print("steps",step+1,"center",v[n//2,n//2])
plt.contourf(xx,xx,v,levels=20);plt.colorbar();plt.axis("equal");plt.show()

## Reflection

1. Why must the FTCS time step shrink quadratically with grid spacing?
2. Why does the heat equation smooth high-frequency modes?
3. What is the difference between an update-based stopping tolerance and a discretization error estimate?